## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# Time-Weighted User-Based Collaborative Filtering

Estimate collaborative relevance from build-window clicks and the query user's available history. Weight interactions by exponential age decay and optional inverse user frequency:

$$w_{ua}=\exp(-\Delta t_{ua}/\tau)\log\frac{N_{users}}{N_{users}(a)}.$$

Normalize weighted user rows before cosine comparison. Shrink similarity by shared-item overlap $o$: $\mathrm{sim}'=\mathrm{sim}\,o/(o+\lambda)$. Normalize the weighted binary-neighbor support to obtain candidate scores. Validation selects neighbor count, shrinkage and decay.

## Temporal experiment

For stack-training scores, fit on the earlier four-day block and score day five. For validation/test, rebuild on the five training days. Do not substitute in-sample scores for stack-training outputs. Missing collaborative predictions retain availability flags and the declared imputation policy. Report full-slate AUC, MRR and nDCG by the source evaluator's cohorts.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import evaluate_scores, history_groups


## 1. Configuration and data support


In [ ]:
import json
import time
from functools import lru_cache
import numpy as np
import pandas as pd
from scipy import sparse

OUT_DIR = str(PATHS["outputs"] / "ubcf_cbf")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
DATA_DIR = str(PATHS["raw"])
os.makedirs(CACHE_DIR, exist_ok=True)
SEED = 42
TRAIN_DAYS = ["2019-11-09", "2019-11-10", "2019-11-11", "2019-11-12", "2019-11-13"]
VAL_DAY = "2019-11-14"
TEST_DAY = "2019-11-15"
PHASES = {
    "stack_train": {"build_days": TRAIN_DAYS[:-1], "score_days": [TRAIN_DAYS[-1]]},
    "val": {"build_days": TRAIN_DAYS, "score_days": [VAL_DAY]},
    "test": {"build_days": TRAIN_DAYS, "score_days": [TEST_DAY]},
}
NEG_K = 4
KS = (5, 10)
WARM_MIN_HISTORY = 5
COLD_SEG = "Cold users (0 history)"
SPARSE_SEG = "Sparse users (1-4 history)"
WARM_SEG = "Warm users (>=5 history)"
USER_SEGS = (COLD_SEG, SPARSE_SEG, WARM_SEG)
OUTPUT_COLS = [
    "impression_id",
    "user_id",
    "news_id",
    "label",
    "score",
    "available",
    "user_cold",
    "item_cold",
]


def _has_parquet() -> bool:
    try:
        import pyarrow  # noqa: F401

        return True
    except ImportError:
        return False


def save_df(df: pd.DataFrame, path_no_ext: str) -> str:
    if _has_parquet():
        path = path_no_ext + ".parquet"
        df.to_parquet(path, index=False)
    else:
        path = path_no_ext + ".pkl"
        df.to_pickle(path)
    return path


def load_df(path_no_ext: str) -> pd.DataFrame:
    if os.path.exists(path_no_ext + ".parquet"):
        return pd.read_parquet(path_no_ext + ".parquet")
    return pd.read_pickle(path_no_ext + ".pkl")


def load_behaviors():
    beh, _ = load_preprocessed(PATHS)
    beh["history"] = beh["history"].str.split()
    beh["date"] = beh["time"].dt.strftime("%Y-%m-%d")
    return beh


def load_candidates(beh):
    rows = beh[["impression_id", "impressions"]].copy()
    rows["impressions"] = rows["impressions"].str.split()
    rows = rows.explode("impressions", ignore_index=True)
    parts = rows["impressions"].str.rsplit("-", n=1, expand=True)
    cand = pd.DataFrame(
        {
            "impression_id": rows["impression_id"].to_numpy(),
            "news_id": parts[0].to_numpy(),
            "label": parts[1].astype(np.int32).to_numpy(),
        }
    )
    cand["candidate_position"] = cand.groupby("impression_id").cumcount()
    return cand.merge(
        beh[["impression_id", "user_id", "time", "date", "split"]],
        on="impression_id",
        how="left",
        validate="many_to_one",
    )


In [ ]:
def build_click_log(beh: pd.DataFrame, cand: pd.DataFrame, build_days) -> pd.DataFrame:
    """All clicks a base model may learn from when built on `build_days`:
       (a) pre-period reading histories of users active in the build window
           (no timestamp in MIND -> stamped at the start of the build window, i.e. oldest)
       (b) impression clicks inside the build window (with their real timestamps).
    Returns one row per (user_id, news_id) with the latest click time."""
    b = beh[beh["date"].isin(build_days)]
    hist = (
        b.sort_values("time")[["user_id", "history"]]
        .drop_duplicates("user_id", keep="last")
        .explode("history")
        .dropna()
        .rename(columns={"history": "news_id"})
    )
    hist["time"] = pd.Timestamp(min(build_days))
    clicks = cand[cand["date"].isin(build_days) & (cand["label"] == 1)][
        ["user_id", "news_id", "time"]
    ]
    log = pd.concat([hist[["user_id", "news_id", "time"]], clicks], ignore_index=True)
    log = log.sort_values("time").drop_duplicates(["user_id", "news_id"], keep="last")
    return log.reset_index(drop=True)


def scoring_histories(beh: pd.DataFrame, score_days) -> pd.DataFrame:
    """Reading history each user brings into the scoring window (earliest impression in the
    window, so no later information is used). Used to fold in users the build log lacks.
    """
    s = beh[beh["date"].isin(score_days)].sort_values("time")
    first = s.drop_duplicates("user_id", keep="first")[["user_id", "history"]]
    h = first.explode("history").dropna().rename(columns={"history": "news_id"})
    return h[["user_id", "news_id"]].reset_index(drop=True)


def user_query_log(
    build_log: pd.DataFrame, beh: pd.DataFrame, score_days, users
) -> pd.DataFrame:
    """Clicks known for each scoring user at scoring time = build-window clicks + history
    they arrive with. History items carry the oldest timestamp of the build window."""
    users = pd.Index(pd.unique(users))
    known = build_log[build_log["user_id"].isin(users)]
    h = scoring_histories(beh, score_days)
    h = h[h["user_id"].isin(users)].copy()
    h["time"] = build_log["time"].min()
    q = pd.concat([known, h], ignore_index=True)
    q = q.sort_values("time").drop_duplicates(["user_id", "news_id"], keep="last")
    return q.reset_index(drop=True)


def phase_candidates(cand: pd.DataFrame, phase: str) -> pd.DataFrame:
    return cand[cand["date"].isin(PHASES[phase]["score_days"])].reset_index(drop=True)


def add_cold_start_flags(
    c: pd.DataFrame, beh: pd.DataFrame, build_log: pd.DataFrame, score_days
) -> pd.DataFrame:
    """Click-history cold-start definitions:
    user_cold = user has 0 clicked articles known at scoring time
                (nothing in the build-window log AND an empty history on arrival)
    item_cold = article has 0 historical clicks in the build window
                (never in any build-window history nor clicked in a build-window impression)
    """
    q = user_query_log(build_log, beh, score_days, c["user_id"])
    warm_users = set(q["user_id"].unique())
    clicked_items = set(build_log["news_id"].unique())
    c = c.copy()
    c["user_cold"] = (~c["user_id"].isin(warm_users)).astype(np.int8)
    c["item_cold"] = (~c["news_id"].isin(clicked_items)).astype(np.int8)
    return c


@lru_cache(maxsize=1)
def user_history_groups() -> pd.Series:
    """impression_id -> reporting group, from the length of the impression's raw MIND `history`
    (same definition as the MLP's cold_user / sparse_user / warm_user slices).
    Evaluation only: unlike user_cold it ignores clicks in the build window, so a user
    in the cold group can still have build-window clicks for the base models to use."""
    n = load_behaviors().set_index("impression_id")["history"].str.len()
    return pd.Series(
        np.select([n == 0, n < WARM_MIN_HISTORY], [COLD_SEG, SPARSE_SEG], WARM_SEG),
        index=n.index,
    )


In [ ]:
def evaluate(df, score_col="score", ks=KS, seed=SEED, by_segment=True):
    return evaluate_scores(df, score_col, ks, seed, by_segment, user_history_groups())


In [ ]:
def save_scores(df: pd.DataFrame, model: str, phase: str, extra_cols=()) -> str:
    """Write a model's scores in the shared format (+ optional model-specific feature cols)."""
    missing = [c for c in OUTPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"missing output columns: {missing}")
    cols = OUTPUT_COLS + [c for c in extra_cols if c not in OUTPUT_COLS]
    out = df[cols].copy()
    out["available"] = out["available"].astype(np.int8)
    os.makedirs(OUT_DIR, exist_ok=True)
    return save_df(out, os.path.join(OUT_DIR, f"scores_{model}_{phase}"))


def load_scores(model: str, phase: str) -> pd.DataFrame:
    return load_df(os.path.join(OUT_DIR, f"scores_{model}_{phase}"))


def standalone_eval(model: str, score_col: str = "score") -> dict:
    """Evaluate a base model on its own: unavailable scores -> training-set mean
    (mean of available scores in the stack_train phase, i.e. scored train-period day).
    """
    tr = load_scores(model, "stack_train")
    fill = (
        float(tr.loc[tr["available"] == 1, score_col].mean())
        if (tr["available"] == 1).any()
        else 0.0
    )
    res = {}
    for ph in ("val", "test"):
        d = load_scores(model, ph)
        d[score_col] = d[score_col].where(d["available"] == 1, fill).fillna(fill)
        res[ph] = evaluate(d, score_col)
    return res


def decay_weights(times: pd.Series, ref_time: pd.Timestamp, tau_days) -> np.ndarray:
    """Exponential time decay exp(-age/tau). tau_days=None -> all clicks weigh 1."""
    if tau_days is None:
        return np.ones(len(times), dtype=np.float32)
    age = (ref_time - times).dt.total_seconds().to_numpy() / 86400.0
    return np.exp(-np.clip(age, 0, None) / tau_days).astype(np.float32)


In [ ]:
beh = load_behaviors()
cand = load_candidates(beh)
print(beh["split"].value_counts().to_dict(), f"| candidate rows: {len(cand):,}")


## 2. Model


In [ ]:
class UBCF:
    def __init__(self, k=100, shrink=10.0, tau_days=None, iuf=True, batch=256):
        self.k, self.shrink, self.tau_days, self.iuf, self.batch = (
            k,
            shrink,
            tau_days,
            iuf,
            batch,
        )

    # ---- build the user-item matrix from the build-window click log ----
    def fit(self, log: pd.DataFrame):
        self.ref_time = log["time"].max()
        self.users = pd.Index(log["user_id"].unique())
        self.items = pd.Index(log["news_id"].unique())
        r = self.users.get_indexer(log["user_id"])
        c = self.items.get_indexer(log["news_id"])
        shape = (len(self.users), len(self.items))
        self.R_bin = sparse.csr_matrix(
            (np.ones(len(log), np.float32), (r, c)), shape=shape
        )
        n_item_users = np.asarray(self.R_bin.sum(axis=0)).ravel()
        self.item_w = (
            np.log(len(self.users) / np.maximum(n_item_users, 1)).astype(np.float32)
            if self.iuf
            else np.ones(len(self.items), np.float32)
        )
        self.R_norm = self._weighted_normed(r, c, log["time"], shape)
        self.RnT = self.R_norm.T.tocsr()
        self.RbT = self.R_bin.T.tocsr()
        return self

    def _weighted_normed(self, r, c, times, shape):
        w = decay_weights(times, self.ref_time, self.tau_days) * self.item_w[c]
        M = sparse.csr_matrix((w, (r, c)), shape=shape)
        norms = np.sqrt(np.asarray(M.multiply(M).sum(axis=1)).ravel())
        return sparse.diags(1.0 / np.maximum(norms, 1e-12)) @ M

    # ---- score (user, candidate) pairs ----
    def score(self, qlog: pd.DataFrame, pairs: pd.DataFrame) -> pd.DataFrame:
        """qlog: clicks known per scoring user (build log + arrival history).
        pairs: rows with user_id, news_id. Returns ubcf_score, available and two
        trust features (neighbour strength, number of neighbours who clicked)."""
        qlog = qlog[qlog["news_id"].isin(self.items)]
        q_users = pd.Index(qlog["user_id"].unique())
        qr = q_users.get_indexer(qlog["user_id"])
        qc = self.items.get_indexer(qlog["news_id"])
        shape = (len(q_users), len(self.items))
        Qn = self._weighted_normed(qr, qc, qlog["time"], shape)
        Qb = sparse.csr_matrix((np.ones(len(qr), np.float32), (qr, qc)), shape=shape)
        self_idx = self.users.get_indexer(q_users)  # -1 if user not in build matrix
        n = len(pairs)
        score_ = np.full(n, np.nan, np.float32)
        strength_ = np.full(n, np.nan, np.float32)
        nclicks_ = np.full(n, np.nan, np.float32)
        pu = q_users.get_indexer(pairs["user_id"])  # -1 -> no clicks at all (cold user)
        pi = self.items.get_indexer(
            pairs["news_id"]
        )  # -1 -> article never clicked (cold item)
        order = np.argsort(pu, kind="stable")  # pairs grouped by query user
        pu_sorted = pu[order]
        k = min(self.k, len(self.users) - 1)
        for start in range(0, len(q_users), self.batch):
            end = min(start + self.batch, len(q_users))
            sims = (Qn[start:end] @ self.RnT).toarray()
            if self.shrink > 0:
                overlap = (Qb[start:end] @ self.RbT).toarray()
                sims *= overlap / (overlap + self.shrink)
            for j, s in enumerate(self_idx[start:end]):
                if s >= 0:
                    sims[j, s] = 0.0  # exclude self
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            top_s = np.take_along_axis(sims, top, axis=1)
            rows = np.repeat(np.arange(end - start), k)
            W = sparse.csr_matrix(
                (top_s.ravel(), (rows, top.ravel())),
                shape=(end - start, len(self.users)),
            )
            Wb = sparse.csr_matrix(
                ((top_s.ravel() > 0).astype(np.float32), (rows, top.ravel())),
                shape=W.shape,
            )
            num = (W @ self.R_bin).toarray()
            cnt = (Wb @ self.R_bin).toarray()
            denom = np.abs(top_s).sum(axis=1)
            lo, hi = np.searchsorted(pu_sorted, [start, end])
            idx = order[lo:hi]  # pairs belonging to this batch
            lu = pu[idx] - start
            strength_[idx] = top_s.mean(axis=1)[lu]
            ok = pi[idx] >= 0
            idx, lu, it = idx[ok], lu[ok], pi[idx][ok]
            d = denom[lu]
            score_[idx] = np.where(d > 0, num[lu, it] / np.where(d > 0, d, 1), np.nan)
            nclicks_[idx] = cnt[lu, it]
        return pd.DataFrame(
            index=pairs.index,
            data={
                "ubcf_score": score_,
                "ubcf_strength": strength_,
                "ubcf_n_clicks": nclicks_,
                "available": (~np.isnan(score_)).astype(np.int8),
            },
        )


In [ ]:
def run_phase(phase, params, cand=cand, beh=beh):
    cfg = PHASES[phase]
    log = build_click_log(beh, cand, cfg["build_days"])
    c = phase_candidates(cand, phase)
    c = add_cold_start_flags(c, beh, log, cfg["score_days"])
    qlog = user_query_log(log, beh, cfg["score_days"], c["user_id"])
    model = UBCF(**params).fit(log)
    s = model.score(qlog, c[["user_id", "news_id"]])
    c = pd.concat([c, s], axis=1)
    c["score"] = c["ubcf_score"]
    return c


## 3. Tune K, shrinkage, time decay on validation
Models are built on train days 1–5 and scored on the validation day. Unavailable pairs get the mean available score so every impression is ranked on its full slate (tie-safe).


In [ ]:
GRID = [
    dict(k=k, shrink=s, tau_days=t)
    for k in (50, 100, 200)
    for s in (0.0, 10.0)
    for t in (None, 2.0)
]
val_runs = []

for p in GRID:
    t0 = time.time()
    v = run_phase("val", p)
    fill = v.loc[v["available"] == 1, "score"].mean()
    v["score"] = v["score"].fillna(fill)
    r = evaluate(v, by_segment=False).iloc[0]
    val_runs.append({**p, **r.to_dict(), "secs": round(time.time() - t0, 1)})
    print(p, f"nDCG@10={r['nDCG@10']:.4f}  AUC={r['AUC']:.4f}")

tuning = pd.DataFrame(val_runs).sort_values("nDCG@10", ascending=False)
BEST = {
    k: (None if pd.isna(v) else v)
    for k, v in tuning.iloc[0][["k", "shrink", "tau_days"]].items()
}
BEST["k"] = int(BEST["k"])
print("best:", BEST)
tuning.head(10)


## 4. Score all phases with the best setting and save (shared output format)


In [ ]:
EXTRA = ["ubcf_strength", "ubcf_n_clicks"]

for phase in PHASES:
    df = run_phase(phase, BEST)
    path = save_scores(df, "ubcf", phase, extra_cols=EXTRA)
    print(
        f"{phase:12s} rows={len(df):,}  available={df['available'].mean():.1%}  "
        f"cold users={df.groupby('impression_id')['user_cold'].first().mean():.1%}  "
        f"cold items={df['item_cold'].mean():.1%}  -> {path}"
    )


## 5. Standalone evaluation (val & test, by cold-start segment)


In [ ]:
res = standalone_eval("ubcf")

for ph, table in res.items():
    table = table[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
    print(f"\n=== UBCF · {ph} ===")
    print(table.round(4).to_string())
    table.to_csv(f"{OUT_DIR}/metrics_ubcf_{ph}.csv")
